# Watch agent — **MEDIUM** (Google Colab)

Characters include **A, N, Z, E, F, H, I, K, M, W, Y**.

**Requirements**
- GPU runtime (T4 or better) for local inference with Qwen2.5-7B + LoRA.
- `ENV_BASE_URL`: your OpenEnv / Space URL serving `LearnHandwritingEnv`.
- `HF_TOKEN`: if the base model or adapter is gated, add a read token (Colab **Secrets** as `HF_TOKEN` or set in code).
- Optional: `LH_REPO_URL` / `LH_REPO_BRANCH` env vars before running the setup cell if you fork the repo.

This notebook is self-contained: it does not link to other notebooks. It clones the project once, then runs inference against your deployed environment. Compare with **`watch_base_model_colab.ipynb`** (set the same `TASK` there).

**Seeing each stroke:** Each step shows **three panels** — target, **cumulative canvas**, and **this step only** (cyan = new ink). **Pause between frames defaults to 2 seconds** so you can narrate (`STROKE_DELAY` env var overrides).


In [ ]:
# @title 1) Install dependencies & clone repository (run once per Colab runtime)
import os, subprocess, sys

REPO_URL = os.environ.get("LH_REPO_URL", "https://github.com/radharamanaa/OpenEnv-Learn-handwriting.git")
BRANCH = os.environ.get("LH_REPO_BRANCH", "").strip()

def sh(cmd: str) -> None:
    subprocess.check_call(cmd, shell=True)

sh(f"{sys.executable} -m pip install -q -U pip")
sh(
    f"{sys.executable} -m pip install -q torch transformers peft accelerate openai pydantic "
    f"python-dotenv opencv-python-headless matplotlib nest_asyncio huggingface_hub 'openenv-core[core]>=0.2.2'"
)

clone_dir = "/content/openenv_lh"
if not os.path.isdir(clone_dir):
    if BRANCH:
        sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {clone_dir}")
    else:
        sh(f"git clone --depth 1 {REPO_URL} {clone_dir}")

PROJ = os.path.join(clone_dir, "learn_handwriting")
if not os.path.isdir(PROJ):
    raise FileNotFoundError(
        f"Expected project at {PROJ}. Set LH_REPO_URL / LH_REPO_BRANCH if your layout differs."
    )
os.chdir(PROJ)
sh(f"{sys.executable} -m pip install -q -e .")
print("Project root:", os.getcwd())


In [ ]:
# @title 2) Config, imports (set MODEL_NAME, ENV_BASE_URL, HF_TOKEN)
import os, sys, asyncio, time

# Repo layout: .../learn_handwriting/visualizations/watch_runner.py
_cwd = os.path.abspath(os.getcwd())
if os.path.basename(_cwd) == "visualizations":
    PROJ_ROOT = os.path.dirname(_cwd)
elif os.path.basename(_cwd) == "learn_handwriting":
    PROJ_ROOT = _cwd
else:
    _fb = "/content/openenv_lh/learn_handwriting"
    PROJ_ROOT = _fb if os.path.isdir(_fb) else _cwd

VIZ = os.path.join(PROJ_ROOT, "visualizations")
os.chdir(VIZ)

os.environ.setdefault("STROKE_DELAY", "2")

# --- Inference: fine-tuned LoRA on Hugging Face (loaded in watch_runner.LocalModelClient) ---
os.environ.setdefault("MODEL_NAME", "abhijeetmishra101/Qwen2.5-7B-Handwriting-GRPO")
os.environ.setdefault("MODEL_REVISION", "main")
# OpenEnv deployment URL (HTTPS, no trailing slash)
os.environ.setdefault("ENV_BASE_URL", "https://YOUR-SPACE.hf.space")

# Optional: Hugging Face token for gated base model / adapter (https://huggingface.co/settings/tokens)
_HF = os.environ.get("HF_TOKEN", "").strip()
if not _HF:
    try:
        from google.colab import userdata  # type: ignore
        _HF = (userdata.get("HF_TOKEN") or "").strip()
    except Exception:
        pass
if _HF:
    os.environ["HF_TOKEN"] = _HF

import nest_asyncio
nest_asyncio.apply()
import matplotlib
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    pass
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output

try:
    from IPython import get_ipython as _get_ipython
    _ipy = _get_ipython()
    if _ipy is not None:
        _ipy.run_line_magic("matplotlib", "inline")
except Exception:
    pass
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(PROJ_ROOT, ".env"), override=False)

from learn_handwriting import LearnHandwritingAction, LearnHandwritingEnv
from watch_runner import (
    get_watch_client,
    StrokeOutput,
    get_stroke,
    _action_str,
    _apply_stroke,
    render_target_character,
    API_KEY,
    API_BASE_URL,
    MODEL_NAME,
    ENV_BASE_URL,
    MAX_STEPS,
)

TASK = "medium"
print(f"Model: {MODEL_NAME}  |  Env: {ENV_BASE_URL}  |  Task: {TASK}")


In [ ]:
# @title Display helper (3 panels: target, full canvas, **this step's stroke only**)
import os as _os

# Default 2s between frames for presentations; override with env STROKE_DELAY.
_frame_delay = float(_os.environ.get("STROKE_DELAY", "2"))


def show_step(
    target,
    canvas,
    step,
    astr,
    reasoning,
    coverage,
    reward,
    integrity,
    done,
    canvas_before=None,
):
    """Colab-friendly: inline backend + third panel shows only new ink from the latest action."""
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
    fig.patch.set_facecolor("#111827")
    for ax in axes:
        ax.set_facecolor("#111827")

    axes[0].imshow(target, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    axes[0].set_title("Target glyph", color="white", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    h, w = canvas.shape[:2]
    bg = np.zeros((h, w, 3), dtype=np.float32)
    bg[..., 0] = 0.06
    bg[..., 1] = 0.09
    bg[..., 2] = 0.16
    c = canvas.astype(np.float32)
    if c.ndim == 2:
        ink = c[..., np.newaxis]
    else:
        ink = c
    vis = bg + ink * np.array([0.95, 0.98, 1.0], dtype=np.float32)
    tgt_dim = np.stack([np.zeros_like(target), target * 0.25, np.zeros_like(target)], axis=-1)
    vis = np.clip(vis + tgt_dim * 0.35, 0, 1)
    axes[1].imshow(vis, interpolation="nearest")
    col = "#4ade80" if coverage >= 90 else "#fb923c" if coverage >= 50 else "#f87171"
    axes[1].set_title(
        f"All strokes  |  step {step}/{MAX_STEPS}  |  {coverage:.1f}%",
        color=col,
        fontsize=12,
        fontweight="bold",
    )
    axes[1].axis("off")

    if canvas_before is not None and step > 0:
        delta = np.clip(canvas.astype(np.float32) - canvas_before.astype(np.float32), 0, 1)
        stroke_rgb = np.stack([delta * 0.15, delta * 0.95, delta * 1.0], axis=-1)
        axes[2].imshow(stroke_rgb, interpolation="nearest")
        axes[2].set_title("This step only (new ink)", color="#22d3ee", fontsize=12, fontweight="bold")
    else:
        axes[2].imshow(np.zeros((h, w, 3)), interpolation="nearest")
        axes[2].set_title("This step only (no stroke yet)", color="#94a3b8", fontsize=12)
    axes[2].axis("off")

    bar = "█" * int(coverage / 5) + "░" * (20 - int(coverage / 5))
    flags = "  INTEGRITY" if integrity else ("  DONE" if done else "")
    fig.text(
        0.5,
        0.02,
        f"Action: {astr}  |  Reward: {reward:.4f}  |  [{bar}]{flags}\n{reasoning[:130]}",
        ha="center",
        color="#e2e8f0",
        fontsize=10,
        fontfamily="monospace",
    )
    plt.tight_layout(rect=[0, 0.12, 1, 1])
    display(fig)
    plt.close(fig)
    time.sleep(_frame_delay)


print("Display helper ready (3-panel stroke view)")


## Watch episode


In [ ]:
# @title Run one episode (GPU recommended)
async def run_episode(task):
    client = get_watch_client()
    env = LearnHandwritingEnv(base_url=ENV_BASE_URL)
    ep = {'task': task, 'character': None, 'steps': [], 'final_score': 0.0, 'success': False}
    obs = None
    try:
        result = await env.reset(task=task)
        obs = result.observation
        char = obs.target_character
        target = render_target_character(char).astype(float)
        canvas = np.zeros((100, 100), dtype=np.int32)
        ep['character'] = char
        print(f'\n{"═"*55}\n  Task: {task.upper()}  Character: "{char}"\n{"═"*55}')
        show_step(target, canvas.astype(float), 0, '(start)', 'Episode started', 0.0, 0.0, False, False)
        history, lm, lw, ink, lr, li = [], 0, 0, getattr(obs,'ink_remaining',9999), 0.0, False
        for step in range(1, MAX_STEPS + 1):
            if result.done:
                break
            stroke = get_stroke(client, step, char, obs.match_percentage, lm, lw, ink, lr, history, li)
            action = LearnHandwritingAction(
                action_type=stroke.action_type, x1=stroke.x1, y1=stroke.y1,
                x2=stroke.x2, y2=stroke.y2, x3=stroke.x3, y3=stroke.y3,
                radius=stroke.radius, rx=stroke.rx, ry=stroke.ry,
            )
            astr = _action_str(stroke)
            canvas_before = canvas.copy()
            canvas = _apply_stroke(canvas, action)
            result = await env.step(action)
            obs = result.observation
            reward = result.reward or 0.0
            lm = obs.pixels_matched_this_stroke
            lw = getattr(obs, 'pixels_wasted_this_stroke', 0)
            ink = getattr(obs, 'ink_remaining', 0)
            li = getattr(obs, 'integrity_violated', False)
            lr = reward
            cov = obs.match_percentage * 100
            show_step(
                target,
                canvas.astype(float),
                step,
                astr,
                stroke.reasoning,
                cov,
                reward,
                li,
                result.done,
                canvas_before=canvas_before.astype(float),
            )
            print(f'  Step {step:>2}: {astr:<35} cov={cov:>5.1f}%  reward={reward:.4f}{" ⚠️" if li else ""}')
            history.append(f'Step {step}: {astr} cov={cov:.1f}% reward={reward:.4f}')
            ep['steps'].append({'step':step,'action':astr,'reasoning':stroke.reasoning,
                'reward':reward,'coverage_pct':cov,'matched_this_step':lm,'wasted_this_step':lw,
                'ink_remaining':ink,'integrity_violated':li})
    finally:
        try:
            await env.close()
        except Exception:
            pass
    score = obs.match_percentage if obs is not None else 0.0
    ep['final_score'] = score
    ep['success'] = score >= 0.90
    print(f'\n  {"SUCCESS" if ep["success"] else "FAILED"}  Final: {score:.1%}  Steps: {len(ep["steps"])}/{MAX_STEPS}')
    return ep

episode = asyncio.run(run_episode(TASK))


## Plots


In [ ]:
# @title Coverage over steps
steps = [s['step'] for s in episode['steps']]
cov   = [s['coverage_pct'] for s in episode['steps']]
integ = [s['step'] for s in episode['steps'] if s['integrity_violated']]
fig, ax = plt.subplots(figsize=(10, 4), facecolor='#111827')
ax.set_facecolor('#1f2937')
ax.plot(steps, cov, marker='o', color='#60a5fa', linewidth=2.5, label='Coverage %')
ax.fill_between(steps, cov, alpha=0.2, color='#60a5fa')
ax.axhline(90, color='#f87171', linestyle='--', linewidth=1.5, label='90% goal')
for s in integ:
    ax.axvline(s, color='#f87171', alpha=0.6, linewidth=1.2, linestyle=':')
ax.set_title(f'Coverage — {TASK.upper()} "{episode["character"]}"  Final: {episode["final_score"]:.1%}',
             color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Step', color='white'); ax.set_ylabel('Coverage (%)', color='white')
ax.tick_params(colors='white'); ax.set_ylim(0, 105)
for spine in ax.spines.values():
    spine.set_edgecolor('#374151')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.2, color='white')
plt.tight_layout()
plt.savefig(f'coverage_{TASK}_{episode["character"]}.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# @title Pixels & rewards per step
steps = [s['step'] for s in episode['steps']]
matched = [s['matched_this_step'] for s in episode['steps']]
wasted  = [s['wasted_this_step']  for s in episode['steps']]
x = np.arange(len(steps)); w = 0.35
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), facecolor='#111827')
for ax in (ax1, ax2):
    ax.set_facecolor('#1f2937'); ax.tick_params(colors='white')
ax1.bar(x-w/2, matched, w, label='Matched', color='#4ade80', alpha=0.85)
ax1.bar(x+w/2, wasted,  w, label='Wasted',  color='#f87171', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels([str(s) for s in steps])
total_m, total_w = sum(matched), sum(wasted)
eff = total_m / max(total_m + total_w, 1) * 100
ax1.set_title(f'Pixels per step  |  Ink efficiency: {eff:.1f}%', color='white', fontsize=11)
ax1.set_xlabel('Step', color='white'); ax1.legend(); ax1.grid(axis='y', alpha=0.2, color='white')
rewards = [s['reward'] for s in episode['steps']]
ax2.bar(steps, rewards, color='#a78bfa', alpha=0.85)
ax2.set_title('Reward per step', color='white', fontsize=11)
ax2.set_xlabel('Step', color='white'); ax2.grid(axis='y', alpha=0.2, color='white')
for spine in ax1.spines.values():
    spine.set_edgecolor('#374151')
for spine in ax2.spines.values():
    spine.set_edgecolor('#374151')
plt.tight_layout()
plt.savefig(f'efficiency_{TASK}_{episode["character"]}.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'\nTotal matched={total_m}  wasted={total_w}  ink_efficiency={eff:.1f}%')
print(f'Total reward={sum(rewards):.4f}  Final score={episode["final_score"]:.1%}')
